In [1]:
import pandas as pd
import re
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

print("All imports successful!")

All imports successful!


In [2]:
# Load cleaned dataset
df = pd.read_csv('../data/headlines.csv')
print("Dataset loaded:", df.shape)
print(df.head())

Dataset loaded: (32000, 2)
                                            headline  clickbait
0                                 Should I Get Bings          1
1      Which TV Female Friend Group Do You Belong In          1
2  The New "Star Wars: The Force Awakens" Trailer...          1
3  This Vine Of New York On "Celebrity Big Brothe...          1
4  A Couple Did A Stunning Photo Shoot With Their...          1


In [3]:
# Initialize stemmer and stopwords
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Step 1: Lowercase everything
    text = text.lower()
    
    # Step 2: Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Step 3: Split into individual words
    tokens = text.split()
    
    # Step 4: Remove stopwords and stem
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words]
    
    # Step 5: Join back into a string
    return ' '.join(tokens)

# Test it on one headline
test = "You Won't Believe What This Celebrity Did Next!!"
print("Original:", test)
print("Cleaned: ", preprocess(test))

Original: You Won't Believe What This Celebrity Did Next!!
Cleaned:  wont believ celebr next


In [4]:
# Apply preprocessing to entire dataset
print("Cleaning all headlines... this may take 30 seconds...")
df['clean_headline'] = df['headline'].apply(preprocess)

print("Done!")
print("\nBefore:", df['headline'][0])
print("After: ", df['clean_headline'][0])

Cleaning all headlines... this may take 30 seconds...
Done!

Before: Should I Get Bings
After:  get bing


In [5]:
# Separate features and labels
X = df['clean_headline']
y = df['clickbait']

# Split 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Training size:", len(X_train))
print("Testing size: ", len(X_test))
print("\nTrain label balance:")
print(y_train.value_counts())

Training size: 25600
Testing size:  6400

Train label balance:
clickbait
0    12801
1    12799
Name: count, dtype: int64


In [6]:
# Create and fit TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 2))

# Fit on training data, transform both
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print("Training matrix shape:", X_train_vec.shape)
print("Testing matrix shape: ", X_test_vec.shape)
print("\nVocabulary size:", len(tfidf.vocabulary_))

# Peek at some features
feature_names = tfidf.get_feature_names_out()
print("\nSample features:", list(feature_names[:10]))

Training matrix shape: (25600, 15000)
Testing matrix shape:  (6400, 15000)

Vocabulary size: 15000

Sample features: ['aaa', 'aaevpc', 'aaron', 'aaron tveit', 'ab', 'abandon', 'abba', 'abbey', 'abc', 'abdelbaset']


In [7]:
# Save the vectorizer
joblib.dump(tfidf, '../model/tfidf_vectorizer.pkl')
print("Vectorizer saved!")

# Save cleaned dataset
df.to_csv('../data/cleaned.csv', index=False)
print("Cleaned dataset saved!")

# Confirm files exist
import os
print("\nFiles in model folder:", os.listdir('../model'))
print("Files in data folder:", os.listdir('../data'))

Vectorizer saved!
Cleaned dataset saved!

Files in model folder: ['tfidf_vectorizer.pkl']
Files in data folder: ['cleaned.csv', 'headlines.csv', 'label_dist.png', 'length_dist.png', 'wc_clickbait.png', 'wc_legit.png']
